In [1]:
# jbu vs strided vs ours (memory, time)
# # Imports
import torch
from torch import nn
torch.cuda.empty_cache()
from torch.nn.functional import interpolate

from hr_dv2 import HighResDV2
import hr_dv2.transform as tr
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from time import time_ns

torch.manual_seed(0)
np.random.seed(0)

from featup.util import norm, unnorm
use_norm = True


/home/ronan/miniconda3/envs/dv2/lib/python3.12/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/home/ronan/miniconda3/envs/dv2/lib/python3.12/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


In [2]:
def load_img(path: str, l: int) -> tuple[torch.Tensor, np.ndarray]:
    temp_img = Image.open(path)
    h, w = temp_img.height, temp_img.width
    #transform = tr.closest_crop(h, w) #tr.get_input_transform(L, L)
    transform = tr.get_input_transform(l, l)
    tensor, img = tr.load_image(path, transform)
    H, W = img.height, img.width
    return tensor, np.array(img)

In [3]:

def get_net(stride: int=4, strided:bool=False) -> nn.Module:
    resize = "nearest-exact" if stride != 14 else "bilinear"
    net = HighResDV2("dinov2_vits14_reg", stride, dtype=torch.float16) #dino_vits8 #dinov2_vits14_reg
    net.interpolation_mode = resize
    net.eval()
    net.cuda()

    fwd_shift, inv_shift = tr.get_shift_transforms([1, 2], 'Moore')
    fwd_flip, inv_flip = tr.get_flip_transforms()
    fwd, inv = tr.combine_transforms(fwd_shift, fwd_flip, inv_shift, inv_flip)
    if strided:
        net.set_transforms([], [])
    else:
        net.set_transforms(fwd_shift, inv_shift)
    return net

In [4]:
def get_jbu() -> torch.nn.Module:
    jbu = torch.hub.load("mhamilton723/FeatUp", 'dinov2', use_norm=True)
    jbu.eval()
    jbu.cuda()
    return jbu

In [5]:
img_tensor, img_arr = load_img('fig_data/1.jpg', 518)
#img_tensor = img_tensor.to(torch.half)
img_tensor = img_tensor.cuda().unsqueeze(0)
pass

In [6]:
@torch.no_grad()
def measure_mem_time(inp: torch.Tensor, model: nn.Module, seq: bool = False) -> tuple[float, float]:
    if type(model) == HighResDV2:
        inp = inp.squeeze(0)

    torch.cuda.reset_peak_memory_stats() # s.t memory is accurate
    torch.cuda.synchronize() # s.t time is accurate
    def _to_MB(x: int) -> float:
        return x / (1024**2)

    def _to_s(t: int) -> float:
        return t / 1e9

    start_m = torch.cuda.max_memory_allocated()
    start_t = time_ns()
    
    if seq:
        model.forward_sequential(inp)
    else:
        model.forward(inp)

    end_m = torch.cuda.max_memory_allocated()
    torch.cuda.synchronize()
    end_t = time_ns()

    return _to_MB(end_m - start_m), _to_s(end_t - start_t)

In [22]:
methods = ["Bilinear", "Strided", "Transforms (batch)", "Transforms (sequential)",]
sizes = [224]  + [224 + 5 * i * 14 for i in range(1, 14) ]
print(sizes)
strided = get_net(4, True)
net_bi = get_net(14, True)
net_tr = get_net(4, False)
# jbu = get_jbu()

[224, 294, 364, 434, 504, 574, 644, 714, 784, 854, 924, 994, 1064, 1134]


Using cache found in /home/ronan/.cache/torch/hub/facebookresearch_dinov2_main
Using cache found in /home/ronan/.cache/torch/hub/facebookresearch_dinov2_main
Using cache found in /home/ronan/.cache/torch/hub/facebookresearch_dinov2_main


In [23]:
out = []
for method in methods:
    torch.cuda.empty_cache()
    print(method)
    times, mems = [], []
    if method == "Strided":
        model = strided
    elif method == "Transforms (batch)" or method == "Transforms (sequential)":
        model = net_tr
    elif method == "Bilinear":
        model = net_bi
    else:
        model = jbu

    for img_size in sizes:
        img_tensor, img_arr = load_img('fig_data/1.jpg', img_size)
        img_tensor = img_tensor.cuda().unsqueeze(0)

        if "seq" in method:
            mem, time = measure_mem_time(img_tensor, model, True)
        elif "Stride" in method:
            mem, time = measure_mem_time(img_tensor, model, True)
        else:
            mem, time = measure_mem_time(img_tensor, model, False)
        times.append(time)
        mems.append(mem)
        img_tensor = img_tensor.cpu()
    out.append(times)
    out.append(mems)
    

Bilinear
Strided
Transforms (batch)
Transforms (sequential)


In [24]:
print(len(out))

8


In [29]:
def format_plot(ax, title: str, xlabel: str, ylabel: str, title_fontsize: int, label_fontsize: int, tick_fontsize: int, legend_fontsize: int ) -> None:
    plt.rcParams["font.family"] = "serif"
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='both')
    ax.set_title(title)
    ax.legend()

    ax.semilogy()

In [26]:
print(len(sizes))
print(len(out))

14
8


In [30]:
plt.style.use('thesis.mplstyle')
fig, axs = plt.subplots(1, 2)
fig.set_size_inches(7, 2.3)

for j, i in enumerate(range(1, 9, 2)):
    data = np.asarray(out[i][1:])
    x = np.asarray(sizes)

    # if j == 3:
    #     data = data / 2.0 #account for fp32 in jbu vs fp16 in ours
    axs[0].plot(x[1:], data, label=methods[j], marker='.', lw=2, ms=10)

for j, i in enumerate(range(0, 8, 2)):
    data = np.asarray(out[i][1:])
    x = np.asarray(sizes)

    axs[1].plot(x[1:], data, label=methods[j], marker='.', lw=2, ms=10)
format_plot(axs[0], "Peak Memory Usage, Stride=4", "Image Length (px)", "Memory (MB)", 20, 16, 14, 14)
format_plot(axs[1], "Time Elapsed, Stride=4", "Image Length (px)", "Time (s)", 20, 16, 14, 14)

SAVE = True
if SAVE:
    plt.savefig('fig_pdf/tt_mem_time_scaling.pdf', dpi=300, bbox_inches='tight', pad_inches=0.01)
    plt.close()